# Day 191 — Month 11 Capstone Part 3: Final Report & GitHub Handoff (Corrected)
**Month 11 · Data Engineering · LogiTrack India Capstone — FINAL DAY**

| Item | Detail |
|------|--------|
| Topic | Consolidate Days 189-190 into a client-deliverable capstone package: architecture diagram, final validated business case, README, GitHub push |
| Builds on | Day 189 (`LogiTrackPipeline`: extract/transform/run_tests/load), Day 190 (`generate_event_log`/`check_conformance`/`monitor`, GitHub Actions) |
| Deliverable | `architecture_diagram.png`, `README.md`, `NRA_Capstone_Business_Case.md`, `pipeline.py`, `test_gate.py`, git push |
| Scoring | 90 pts + 10 star bonus |
| Environment | Google Colab |

---
### Capstone roadmap
| Day | Focus |
|---|---|
| 189 | Unified pipeline class, test suite, S3-partitioned load, run-summary artifact — **90/90+10★** |
| 190 | GitHub Actions scheduled workflow + test gate + process-mining monitoring step — **90/90+10★** |
| **191 (today)** | Final capstone report + GitHub push — closes Month 11 |

> **Correction note:** an earlier draft of this notebook was built against placeholder raw data that did not match your actual Day 189/190 dataset (different carrier list, missing `weight_kg`/`revenue_inr`/`status`/`ship_date` columns). This version uses the exact locked raw-data cell from your real Day 189/190 notebooks, re-executed here, so every number below is verified against your actual capstone data — not a stand-in.

> **Critical rule enforced today (carried from the Day 190 lesson):**
> Every number in the final report is pulled from a cell executed in *this* notebook, right now — not recalled from Day 189's or Day 190's memory summary.

## Section 1 — Locked Raw Data (DO NOT MODIFY)

Identical to the locked cell in Day 189 and Day 190 — seed=181, same columns, same 8 injected duplicates.

In [1]:
# ============================================================
# LOCKED RAW DATA — DO NOT MODIFY THIS CELL
# LogiTrack India, seed=181 (identical to Days 189-190)
# ============================================================
import pandas as pd
import numpy as np
import os, io, json
from datetime import datetime, timedelta

os.environ.setdefault('AWS_DEFAULT_REGION', 'ap-south-1')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'testing')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'testing')
os.environ.setdefault('AWS_SECURITY_TOKEN', 'testing')
os.environ.setdefault('AWS_SESSION_TOKEN', 'testing')

import boto3
from moto import mock_aws

np.random.seed(181)
n = 530

carriers = ['BlueDart', 'Delhivery', 'DTDC', 'XpressBees', 'Ecom Express']
cities_origin = ['Mumbai', 'Delhi', 'Bengaluru', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Ahmedabad']
cities_dest = ['Jaipur', 'Lucknow', 'Bhopal', 'Patna', 'Surat', 'Nagpur', 'Visakhapatnam', 'Indore']
statuses = ['Delivered', 'In Transit', 'Delayed', 'Returned']

shipment_ids = [f'LT{str(i).zfill(5)}' for i in range(1, n + 1)]
carrier_col = np.random.choice(carriers, n)
origin_col = np.random.choice(cities_origin, n)
destination_col = np.random.choice(cities_dest, n)
weight_col = np.round(np.random.uniform(0.5, 50.0, n), 2)
days_col = np.random.randint(1, 25, n)
status_col = np.random.choice(statuses, n, p=[0.6, 0.2, 0.15, 0.05])
revenue_col = np.round(weight_col * np.random.uniform(20, 80, n), 2)

base_date = datetime(2024, 1, 1)
ship_dates = [base_date + timedelta(days=int(x)) for x in np.random.randint(0, 365, n)]
ship_date_col = [d.strftime('%Y-%m-%d') for d in ship_dates]

df_base = pd.DataFrame({
    'shipment_id': shipment_ids, 'carrier': carrier_col, 'origin': origin_col,
    'destination': destination_col, 'weight_kg': weight_col, 'days_in_transit': days_col,
    'status': status_col, 'revenue_inr': revenue_col, 'ship_date': ship_date_col
})

# inject 8 duplicate rows — same corruption pattern as Days 181/186/189/190
dup_indices = np.random.choice(range(n), 8, replace=False)
dups = df_base.iloc[dup_indices].copy()
df_raw = pd.concat([df_base, dups], ignore_index=True)

os.makedirs('capstone_raw', exist_ok=True)
raw_path = 'capstone_raw/logitrack_raw.csv'
df_raw.to_csv(raw_path, index=False)

print(f"Raw rows: {len(df_raw)}")
df_raw.head(3)

Raw rows: 538


,shipment_id,carrier,origin,destination,weight_kg,days_in_transit,status,revenue_inr,ship_date
0,LT00001,Delhivery,Pune,Jaipur,35.28,7,Delayed,2782.94,2024-08-16
1,LT00002,Delhivery,Kolkata,Nagpur,7.04,3,Delivered,198.51,2024-07-16
2,LT00003,Delhivery,Pune,Nagpur,45.43,21,Delayed,2385.01,2024-03-11


## Section 2 — Concept Notes

### What a "capstone final report" actually is in client work
Every prior Month 11 day produced a working component. None of those components are useful to a client or a hiring manager until they're packaged into three artifacts:

1. **An architecture diagram** — the 30-second version. Shows what talks to what, and where the safety gates sit (test gate, conformance threshold).
2. **A README** — the "can a stranger run this" test. If a new engineer can't get from clone to running pipeline using only your README, the README failed.
3. **A business case** — the "why should anyone care" version, written in NRA (Number, Reason, Action) format, that survives a skeptical read.

### Why re-verify instead of reuse
Day 190 caught a real failure mode: naming a specific carrier as "worst" based on a *previous* run's numbers, when the current run's actual output showed a different carrier. This notebook itself was rebuilt for the same reason — its first draft used placeholder data instead of your real Day 189/190 dataset. **Every number in a final report is pulled from a cell you just executed, in the notebook you're currently looking at — never from a memory summary, even Claude's own.**

### Architecture diagrams: what belongs in one
- Data flow direction (arrows show movement, not just "connection")
- Decision points (the test gate, the conformance threshold) drawn distinctly from normal flow steps
- Failure paths (what happens when a gate fails), not just the happy path
- Enough labels that someone unfamiliar with the code can explain it back to you

### README structure that survives a real audit
Results table with numbers a reader can independently check -> tech stack -> exact run commands -> repo structure. Anything else (long narrative, marketing language) is noise a technical reader skips.

### Interview framing
*"Walk me through a project where you had to explain technical work to a non-technical stakeholder."*
> "In my LogiTrack capstone, I separated the two audiences into two documents: the README for someone who's going to run or maintain the code, and a Number-Reason-Action business case for someone who's deciding whether the finding is worth acting on. The business case format forces me to state a real number, the actual causal mechanism, and a specific action — and just as importantly, it forces me to say when I *don't* have enough evidence to name a carrier-specific cause, which happened here: my worst-conformance carrier turned out to be explained entirely by a delay-linked probability that applies identically to every carrier, not a carrier-specific defect."

## Section 3 — Practice Tasks

**T1 (15 pts):** Re-run the ETL pipeline + 6-assertion test gate on the locked raw data above. Print: raw/clean row counts, high-delay count/%, revenue at risk, carrier ranking by average transit time.

**T2 (15 pts):** Re-run the process-mining monitor with a fixed seed (document it). Print: event/case counts, conformance %, per-carrier conformance breakdown.

**T3 (20 pts):** Generate an architecture diagram covering: raw data -> pipeline (extract/transform/test gate/load) -> S3 -> run-summary -> process-mining monitor -> GitHub Actions. Show both the pass path and the fail/alert path.

**T4 (20 pts):** Write `README.md` **to disk** (not just a notebook cell) — results table matching T1/T2 output, tech stack, run commands, repo structure.

**T5 (20 pts):** Write the final NRA Capstone Business Case — 3 bullets, numbers from T1/T2 of *this* notebook only. Check every Reason against the generator code before writing it.

**Bonus (★10):** git commit + push checklist to `Month11-DataEngineering-Portfolio` on `master`.

### T1 — ETL re-validation (skeleton)

### T2 — Process-mining monitor re-validation (skeleton)

### T3 — Architecture diagram (skeleton)

### T4 — README.md (skeleton)

### T5 — NRA Capstone Business Case (skeleton)

## Section 4 — Submission

### T1 — ETL re-validation

In [7]:
# ============================================================
# ANSWER KEY — T1
# ============================================================
df_raw2 = pd.read_csv(raw_path)
print(f"Extracted rows: {len(df_raw2)}")

df_clean = df_raw2.drop_duplicates().reset_index(drop=True)
df_clean['delay_flag'] = df_clean['days_in_transit'].apply(lambda x: 'High Delay' if x > 14 else 'On Track')
df_clean['high_delay'] = df_clean['delay_flag'] == 'High Delay'

total_rows = len(df_clean)
high_delay_count = int(df_clean['high_delay'].sum())
high_delay_pct = round(high_delay_count / total_rows * 100, 2)
revenue_at_risk = round(df_clean[df_clean['high_delay']]['revenue_inr'].sum(), 2)
avg_transit_by_carrier = df_clean.groupby('carrier')['days_in_transit'].mean().round(2).sort_values(ascending=False)
high_delay_share = df_clean.groupby('carrier')['high_delay'].mean().mul(100).round(2).sort_values(ascending=False)

checks = [
    ('raw_row_count', len(df_raw2) == 538),
    ('clean_row_count', total_rows == 530),
    ('no_duplicates', df_clean.duplicated().sum() == 0),
    ('no_nulls', df_clean.isnull().sum().sum() == 0),
    ('shipment_id_unique', df_clean['shipment_id'].is_unique),
    ('revenue_non_negative', (df_clean['revenue_inr'] >= 0).all()),
]
all_pass = True
for name, passed in checks:
    print(f"  {'✓ PASS' if passed else '✗ FAIL'}: {name}")
    if not passed:
        all_pass = False
print("✅ All tests passed" if all_pass else "❌ Some tests failed")

print(f"\nClean rows               : {total_rows}")
print(f"High-delay shipments     : {high_delay_count} ({high_delay_pct}%)")
print(f"Revenue at risk (Rs)     : {revenue_at_risk:,.2f}")
print("\nCarrier avg transit days (worst first):")
print(avg_transit_by_carrier)
print("\nCarrier high-delay share % (worst first):")
print(high_delay_share)

Extracted rows: 538
  ✓ PASS: raw_row_count
  ✓ PASS: clean_row_count
  ✓ PASS: no_duplicates
  ✓ PASS: no_nulls
  ✓ PASS: shipment_id_unique
  ✓ PASS: revenue_non_negative
✅ All tests passed

Clean rows               : 530
High-delay shipments     : 234 (44.15%)
Revenue at risk (Rs)     : 305,612.65

Carrier avg transit days (worst first):
carrier
Delhivery       13.13
DTDC            13.10
BlueDart        12.99
XpressBees      12.92
Ecom Express    11.93
Name: days_in_transit, dtype: float64

Carrier high-delay share % (worst first):
carrier
BlueDart        50.00
XpressBees      46.02
DTDC            44.72
Delhivery       44.66
Ecom Express    35.05
Name: high_delay, dtype: float64


**Verified output (this run):** 538 raw -> 530 clean · 234 high-delay (44.15%) · Rs 3,05,612.65 revenue at risk · worst carrier by avg transit = Delhivery (13.13 days) · worst by high-delay share = BlueDart (50.00%) · 6/6 gate checks pass.

### T2 — Process-mining monitor re-validation (fixed seed=190, matching Day 190)

In [8]:
# ============================================================
# ANSWER KEY — T2
# ============================================================
np.random.seed(190)
canonical = ['Order Placed', 'Picked Up', 'In Transit Scan', 'Out for Delivery', 'Delivered']
middle = canonical[1:-1]

events = []
for _, row in df_clean.iterrows():
    case_id = row['shipment_id']
    carrier = row['carrier']
    break_prob = 0.45 if row['high_delay'] else 0.15
    timestamp = datetime.strptime(row['ship_date'], '%Y-%m-%d')
    sequence = canonical.copy()
    if np.random.random() < break_prob:
        break_type = np.random.choice(['skip', 'reorder'])
        if break_type == 'skip':
            idx = np.random.randint(0, len(middle))
            sequence.remove(middle[idx])
        else:
            idx = np.random.randint(0, len(middle) - 1)
            p1, p2 = sequence.index(middle[idx]), sequence.index(middle[idx + 1])
            sequence[p1], sequence[p2] = sequence[p2], sequence[p1]
    for activity in sequence:
        timestamp += timedelta(hours=np.random.uniform(2, 30))
        events.append({'case_id': case_id, 'carrier': carrier, 'activity': activity, 'timestamp': timestamp})

df_events = pd.DataFrame(events)
print(f"Total events: {len(df_events)}")
print(f"Total cases: {df_events['case_id'].nunique()}")

conformant, non_conformant = [], []
for case_id, group in df_events.groupby('case_id'):
    activities = group.sort_values('timestamp')['activity'].tolist()
    if set(activities) != set(canonical):
        non_conformant.append(case_id)
        continue
    order = [canonical.index(a) for a in activities]
    (conformant if all(order[i] < order[i+1] for i in range(len(order)-1)) else non_conformant).append(case_id)

total_cases = len(conformant) + len(non_conformant)
conformance_rate = round(len(conformant) / total_cases * 100, 2)
print(f"\nConformance rate: {conformance_rate}%")
print(f"Non-conformant cases: {len(non_conformant)}")

carrier_stats = {}
for carrier, group in df_events.groupby('carrier'):
    conf, nonc = 0, 0
    for case_id, g in group.groupby('case_id'):
        acts = g.sort_values('timestamp')['activity'].tolist()
        if set(acts) != set(canonical):
            nonc += 1
            continue
        order = [canonical.index(a) for a in acts]
        if all(order[i] < order[i+1] for i in range(len(order)-1)):
            conf += 1
        else:
            nonc += 1
    tot = conf + nonc
    carrier_stats[carrier] = {'total': tot, 'conformant': conf, 'non_conformant': nonc,
                               'conformance_rate': round(conf/tot*100, 2) if tot else 0}

carrier_df = pd.DataFrame(carrier_stats).T.sort_values('conformance_rate')
print("\nCarrier-level conformance breakdown (worst first):")
print(carrier_df)

monitor_status = "ALERT — below 80% threshold" if conformance_rate < 80 else "OK"
print(f"\nMonitor status: {monitor_status}")

Total events: 2580
Total cases: 530



Conformance rate: 71.32%
Non-conformant cases: 152



Carrier-level conformance breakdown (worst first):
              total  conformant  non_conformant  conformance_rate
BlueDart       94.0        58.0            36.0             61.70
XpressBees    113.0        81.0            32.0             71.68
Delhivery     103.0        75.0            28.0             72.82
Ecom Express   97.0        71.0            26.0             73.20
DTDC          123.0        93.0            30.0             75.61

Monitor status: ALERT — below 80% threshold


**Verified output (this run, seed=190):** 530 cases · conformance 71.32% (378/530 conformant, 152 non-conformant) · ALERT (below 80%) · worst carrier by conformance = BlueDart (61.70%, 58/94).

### T3 — Architecture diagram

In [9]:
# ============================================================
# ANSWER KEY — T3 (graphviz)
# ============================================================
import graphviz

dot = graphviz.Digraph('LogiTrackCapstoneArchitecture', format='png')
dot.attr(rankdir='LR', bgcolor='white', fontname='Arial')
dot.attr('node', fontname='Arial', fontsize='11', style='filled')

DARK_BLUE, GREEN, YELLOW, RED, LIGHT = '#1F3864', '#2E7D32', '#F9A825', '#C62828', '#D9E2F3'

dot.node('raw', 'Raw CSV\n(seed=181, 538 rows)', shape='folder', fillcolor=LIGHT, color=DARK_BLUE)
with dot.subgraph(name='cluster_pipeline') as c:
    c.attr(label='LogiTrackPipeline (Day 189)', color=DARK_BLUE, fontcolor=DARK_BLUE)
    c.node('extract', 'extract()', shape='box', fillcolor=LIGHT, color=DARK_BLUE)
    c.node('transform', 'transform()\ndedupe -> 530 rows', shape='box', fillcolor=LIGHT, color=DARK_BLUE)
    c.node('tests', 'run_tests()\n6-assertion gate', shape='diamond', fillcolor='#FFF3CD', color=YELLOW)
    c.node('load', 'load()\nS3 partitioned', shape='box', fillcolor=LIGHT, color=DARK_BLUE)
    c.edge('extract', 'transform'); c.edge('transform', 'tests')
    c.edge('tests', 'load', label='PASS', color=GREEN, fontcolor=GREEN)
    c.node('blocked', 'BLOCKED (exit 1)', shape='octagon', fillcolor='#F8D7DA', color=RED, fontcolor=RED)
    c.edge('tests', 'blocked', label='FAIL', color=RED, fontcolor=RED)
dot.edge('raw', 'extract')
dot.node('s3', 'S3 (moto)\n5 objects', shape='cylinder', fillcolor=LIGHT, color=DARK_BLUE)
dot.edge('load', 's3')
dot.node('summary', 'run_summary.json', shape='note', fillcolor=LIGHT, color=DARK_BLUE)
dot.edge('load', 'summary')
with dot.subgraph(name='cluster_monitor') as m:
    m.attr(label='Process-Mining Monitor (Day 190)', color=DARK_BLUE, fontcolor=DARK_BLUE)
    m.node('eventlog', 'generate_event_log()\nseed=190', shape='box', fillcolor=LIGHT, color=DARK_BLUE)
    m.node('conformance', 'check_conformance()', shape='box', fillcolor=LIGHT, color=DARK_BLUE)
    m.node('gate2', 'monitor()\n80% threshold', shape='diamond', fillcolor='#FFF3CD', color=YELLOW)
    m.edge('eventlog', 'conformance'); m.edge('conformance', 'gate2')
dot.edge('summary', 'eventlog', style='dashed')
dot.node('alert', 'ALERT\n71.32% conformance\nworst: BlueDart 61.70%', shape='octagon', fillcolor='#F8D7DA', color=RED, fontcolor=RED)
dot.edge('gate2', 'alert', label='below threshold', color=RED, fontcolor=RED)
dot.node('gha', 'GitHub Actions\ncron + dispatch', shape='box', fillcolor='#E8F5E9', color=GREEN)
dot.edge('gha', 'extract', style='dashed', label='triggers run')
dot.edge('gate2', 'gha', style='dashed', label='artifact upload')

dot.render('architecture_diagram', cleanup=True)
print("Saved architecture_diagram.png")

Saved architecture_diagram.png


### T4 — README.md (written to disk)

In [10]:
# ============================================================
# ANSWER KEY — T4 — actually writes the file, not just a cell
# ============================================================
readme_content = f'''# LogiTrack India - Data Engineering Capstone

## Results Summary (verified run, seed=181 / seed=190)

| Metric | Value |
|--------|-------|
| Raw rows | {len(df_raw2)} |
| Clean rows | {total_rows} |
| High-delay shipments | {high_delay_count} ({high_delay_pct}%) |
| Revenue at risk | Rs {revenue_at_risk:,.2f} |
| Conformance rate | {conformance_rate}% |
| Non-conformant cases | {len(non_conformant)} |

**Worst carrier by transit time:** {avg_transit_by_carrier.index[0]} ({avg_transit_by_carrier.iloc[0]} days)
**Worst carrier by conformance:** {carrier_df.index[0]} ({carrier_df.iloc[0]["conformance_rate"]}% conformant)

> Note: high-delay threshold is `days_in_transit > 14`, a placeholder pending
> a real client SLA. Carrier-level differences above are explained by the
> delay-linked break probability, not a carrier-specific mechanism -- see
> NRA_Capstone_Business_Case.md.

---

## Tech Stack
Python 3.11+, pandas, numpy, boto3 + moto (S3 mocking), graphviz, GitHub Actions

## How to Run
```
git clone https://github.com/deepanshu0110/Month11-DataEngineering-Portfolio.git
cd Month11-DataEngineering-Portfolio
pip install -r requirements.txt
python pipeline.py
python test_gate.py
```

## Repository Structure
```
.
├── notebooks/Day189...ipynb, Day190...ipynb, Day191...ipynb
├── pipeline.py
├── test_gate.py
├── .github/workflows/logitrack_monitor.yml
├── architecture_diagram.png
├── NRA_Capstone_Business_Case.md
└── README.md
```
'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)
print("README.md written to disk.")

README.md written to disk.


### T5 — NRA Capstone Business Case (written to disk)

In [11]:
# ============================================================
# ANSWER KEY — T5
# ============================================================
worst_transit_carrier = avg_transit_by_carrier.index[0]
worst_transit_days = avg_transit_by_carrier.iloc[0]
best_transit_carrier = avg_transit_by_carrier.index[-1]
best_transit_days = avg_transit_by_carrier.iloc[-1]
worst_conf_carrier = carrier_df.index[0]
worst_conf_rate = carrier_df.iloc[0]['conformance_rate']
worst_conf_total = int(carrier_df.iloc[0]['total'])
worst_conf_conformant = int(carrier_df.iloc[0]['conformant'])
worst_delay_share_carrier = high_delay_share.index[0]
worst_delay_share_pct = high_delay_share.iloc[0]

nra_bullets = f'''# Month 11 Capstone - Final Business Case Report

### Bullet 1 - Delay exposure
Number: {high_delay_count} of {total_rows} shipments ({high_delay_pct}%) are flagged High Delay (>14 days transit), putting Rs {revenue_at_risk:,.2f} in revenue at risk.
Reason: days_in_transit is drawn independently per shipment with no carrier-linked parameter in the generator -- the share above a 14-day cutoff is a function of the distribution's width relative to the cutoff, not a designed failure mode.
Action: Owner - pipeline lead. Confirm the 14-day cutoff against the client's real SLA (currently a placeholder) before this threshold appears in any client-facing report.

### Bullet 2 - Carrier transit-time ranking
Number: {worst_transit_carrier} shows the highest average transit time at {worst_transit_days} days vs. {best_transit_carrier}'s lowest at {best_transit_days} days.
Reason: days_in_transit is generated independently of carrier; this ranking reflects sampling variance across ~94-123 shipments per carrier, not a measured operational difference.
Action: Owner - carrier relationship manager. Do not reallocate volume based on this ranking; re-test against >=3 months of real transit logs first.

### Bullet 3 - Process conformance monitor
Number: Overall conformance is {conformance_rate}% ({total_cases - len(non_conformant)}/{total_cases}), below the 80% threshold. {worst_conf_carrier} has the lowest carrier-level conformance at {worst_conf_rate}% ({worst_conf_conformant}/{worst_conf_total}), and also the highest high-delay share at {worst_delay_share_pct}%.
Reason: The rework/break event is injected via a delay-linked probability (45% if high-delay, 15% if not) that is carrier-independent in the generator. {worst_conf_carrier}'s lower conformance is explained by its high-delay share feeding into that same mechanism -- not a {worst_conf_carrier}-specific process failure.
Action: Owner - monitoring lead. Ship the {conformance_rate}% alert as-is. Do not target {worst_conf_carrier} specifically; investigate root causes of high-delay shipments across all carriers, since fixing the delay rate fixes conformance for every carrier through the same mechanism. Weekly review with operations; escalate if below 80% for 2 consecutive weeks.
'''

print(nra_bullets)
with open('NRA_Capstone_Business_Case.md', 'w', encoding='utf-8') as f:
    f.write(nra_bullets)
print("Business case saved as NRA_Capstone_Business_Case.md")

# Month 11 Capstone - Final Business Case Report

### Bullet 1 - Delay exposure
Number: 234 of 530 shipments (44.15%) are flagged High Delay (>14 days transit), putting Rs 305,612.65 in revenue at risk.
Reason: days_in_transit is drawn independently per shipment with no carrier-linked parameter in the generator -- the share above a 14-day cutoff is a function of the distribution's width relative to the cutoff, not a designed failure mode.
Action: Owner - pipeline lead. Confirm the 14-day cutoff against the client's real SLA (currently a placeholder) before this threshold appears in any client-facing report.

### Bullet 2 - Carrier transit-time ranking
Number: Delhivery shows the highest average transit time at 13.13 days vs. Ecom Express's lowest at 11.93 days.
Reason: days_in_transit is generated independently of carrier; this ranking reflects sampling variance across ~94-123 shipments per carrier, not a measured operational difference.
Action: Owner - carrier relationship manager. Do

### Bonus (★10) — git push checklist

In [12]:
checklist = '''
git init                      # first time only
git checkout -b master        # GitHub defaults to main; this repo uses master
git add pipeline.py test_gate.py README.md NRA_Capstone_Business_Case.md \
        architecture_diagram.png .github/workflows/logitrack_monitor.yml \
        notebooks/Day189_Capstone_Part1_Unified_Pipeline.ipynb \
        notebooks/Day190_Capstone_Part2_Monitoring.ipynb \
        notebooks/Day191_Capstone_Final_Report.ipynb
git commit -m "feat: Day191 - Capstone Final Report"
git remote add origin https://github.com/deepanshu0110/Month11-DataEngineering-Portfolio.git
git push -u origin master
'''
print(checklist)


git init                      # first time only
git checkout -b master        # GitHub defaults to main; this repo uses master
git add pipeline.py test_gate.py README.md NRA_Capstone_Business_Case.md         architecture_diagram.png .github/workflows/logitrack_monitor.yml         notebooks/Day189_Capstone_Part1_Unified_Pipeline.ipynb         notebooks/Day190_Capstone_Part2_Monitoring.ipynb         notebooks/Day191_Capstone_Final_Report.ipynb
git commit -m "feat: Day191 - Capstone Final Report"
git remote add origin https://github.com/deepanshu0110/Month11-DataEngineering-Portfolio.git
git push -u origin master



## Section 5 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| T1 — ETL re-validation | 15 | Correct dedupe/flag logic, all 6 gate checks with real (non-tautological) assertions, numbers match cell output |
| T2 — Monitor re-validation | 15 | Seed fixed and documented, conformance % + carrier breakdown printed correctly |
| T3 — Architecture diagram | 20 | Covers full system incl. both gates and both fail/alert paths, legible labels |
| T4 — README.md | 20 | Actually written to disk; results table numbers match T1/T2 output exactly |
| T5 — NRA Business Case | 20 | Numbers from this run's output only; Reason checked against generator code, no fabricated mechanism; Action has a named owner |
| Bonus — git push checklist | ★10 | Correct commit convention, correct branch (`master`), all deliverable files included |
| **Total** | **90 + ★10** | |

## Month 11 — Complete

Days 181-191, all eleven days scored. This closes the Data Engineering month. Month 12 begins with the final capstone integration across the full 12-month curriculum, the Netherlands application package (CV, motivation letter draft, Holland Scholarship + OTS), and the IELTS milestone.